In [1]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [11]:
def extract_text(response) -> str:
    """ Normalizes an LLM response's content into a plain string.

    Some models (e.g. Gemini) can return `content` as a list of content
    blocks (dicts with a 'text' key, or plain strings) instead of a str.
    """
    if not response:
        return "No summary available."

    content = response.content
    if isinstance(content, list):
        text = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    else:
        text = content

    return text.strip() if text else "No summary available."

In [12]:
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="You are a helpful assistant. Answer all questions to the best of your ability."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [13]:
ai_msg = chain.invoke(
    {
        "messages": [
            HumanMessage(content="Can you explain Dynamic Programming under 50 words?")
        ],
    }
)

# print(ai_msg.content)
print(extract_text(ai_msg))

Dynamic Programming is an optimization technique that solves complex problems by breaking them into simpler, overlapping subproblems. It computes each subproblem's solution once and stores the result—using memoization or tabulation—to avoid redundant calculations. This transforms inefficient recursive processes into fast, high-performance algorithms.


In [16]:
chat_history = [
    HumanMessage(content="Can you explain Dynamic Programming under 50 words?"),
    AIMessage(content=extract_text(ai_msg))
]

chat_history

[HumanMessage(content='Can you explain Dynamic Programming under 50 words?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Dynamic Programming is an optimization technique that solves complex problems by breaking them into simpler, overlapping subproblems. It computes each subproblem's solution once and stores the result—using memoization or tabulation—to avoid redundant calculations. This transforms inefficient recursive processes into fast, high-performance algorithms.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [17]:
def interrogate_model(chat_history: list, preserve_chat_history=True):
    current_message = [HumanMessage(content="What did I ask you earlier?")]
    chat_messages = []

    if preserve_chat_history:
        chat_messages = chat_history + current_message
    else:
        chat_messages = current_message


    ai_msg = chain.invoke(
        {
            "messages": chat_messages
        }
    )

    return extract_text(ai_msg)

In [ ]:
interrogate_model(chat_history, preserve_chat_history=True)

"In this current conversation, this is the first question you've asked me. \n\nAs an AI, I don't have a long-term memory of our past conversations once a session is ended or cleared. If you asked me something in a different chat window or a previous session, I wouldn't be able to recall it.\n\nIf you were thinking of a specific topic or question, feel free to ask it again or give me a few keywords, and I'll do my best to help!"

In [ ]:
def summarize_chat_history(chat_history: list):
    """ Summarizes the chat history when the message count is greater than 4."""

    if len(chat_history) >= 4:
        summary_prompt = (
            "Distill the above chat messages into a single summary message. "
            "Include as many specific details as you can."
        )
        print(chat_history)
        summary_message = chain.invoke(
            chat_history + [HumanMessage(content=summary_prompt)]
        )

        # return summary_message.content
        return extract_text(summary_message)
    
    return ""

In [21]:
def interrogate_model_with_summary(query: str, chat_history: list, debug_mode=False):
    current_message = [HumanMessage(content=query)]
    chat_messages = []

    if len(chat_history) >= 4:
        print("Summarizing your content...")
        summary_message = summarize_chat_history(chat_history)
        chat_messages = [AIMessage(content=summary_message)] + current_message
    else:
        chat_messages = chat_history + current_message
    
    if debug_mode:
        print("Messages sent to the model:")
        print(chat_messages, "\n")

    ai_msg = chain.invoke(
        {
            "messages": chat_messages
        }
    )

    return AIMessage(content=extract_text(ai_msg))

In [22]:
def chat_with_model(query: str, chat_history: list, debug_mode=False):
    query_response = interrogate_model_with_summary(query, chat_history, debug_mode)
    
    # Append the query and response to the chat history
    chat_history.append(HumanMessage(content=query))
    chat_history.append(query_response)
    
    return query_response, chat_history

In [23]:
chat_history = []

query1 = "Can you explain Dynamic Programming under 50 words?"
query_response1, chat_history = chat_with_model(query1, chat_history, debug_mode=True)

print(query_response1.content)

Messages sent to the model:
[HumanMessage(content='Can you explain Dynamic Programming under 50 words?', additional_kwargs={}, response_metadata={})] 

Dynamic Programming solves complex problems by breaking them into simpler, overlapping subproblems. It stores the results of these subproblems (using memoization or tabulation) to avoid repeating calculations. This approach significantly improves efficiency for problems featuring optimal substructure and redundant tasks.


In [24]:
query2 = "Can you explain Graph data structures under 50 words?"
query_response2, chat_history = chat_with_model(query2, chat_history, debug_mode=True)

print(query_response2.content)

Messages sent to the model:
[HumanMessage(content='Can you explain Dynamic Programming under 50 words?', additional_kwargs={}, response_metadata={}), AIMessage(content='Dynamic Programming solves complex problems by breaking them into simpler, overlapping subproblems. It stores the results of these subproblems (using memoization or tabulation) to avoid repeating calculations. This approach significantly improves efficiency for problems featuring optimal substructure and redundant tasks.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Can you explain Graph data structures under 50 words?', additional_kwargs={}, response_metadata={})] 

Graphs consist of nodes (vertices) and connections (edges) that represent relationships between objects. They can be directed or undirected, weighted or unweighted. Widely used for social networks and navigation, graphs are typically implemented using adjacency lists or matrices to store and trave

In [25]:
query3 = "Can you tell me what did I ask till now?"
query_response3, chat_history = chat_with_model(query3, chat_history, debug_mode=True)

print(query_response3.content)

Summarizing your content...
[HumanMessage(content='Can you explain Dynamic Programming under 50 words?', additional_kwargs={}, response_metadata={}), AIMessage(content='Dynamic Programming solves complex problems by breaking them into simpler, overlapping subproblems. It stores the results of these subproblems (using memoization or tabulation) to avoid repeating calculations. This approach significantly improves efficiency for problems featuring optimal substructure and redundant tasks.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Can you explain Graph data structures under 50 words?', additional_kwargs={}, response_metadata={}), AIMessage(content='Graphs consist of nodes (vertices) and connections (edges) that represent relationships between objects. They can be directed or undirected, weighted or unweighted. Widely used for social networks and navigation, graphs are typically implemented using adjacency lists or matrices t